In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
import numpy as np
from tqdm import tqdm

# PennyLane imports
import pennylane as qml
from pennylane.qnn import TorchLayer

ModuleNotFoundError: No module named 'pennylane'

In [ ]:
train_dir = "dataset/train"
val_dir   = "dataset/test"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch using:", device)

batch_size = 16
num_epochs = 10
pca_components = 12   # = qubits
n_qubits = pca_components
n_variational_layers = 2
num_classes = 2

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

train_ds = datasets.ImageFolder(train_dir, transform=transform)
val_ds   = datasets.ImageFolder(val_dir, transform=transform)

train_loader_img = DataLoader(train_ds, batch_size=batch_size, shuffle=False)
val_loader_img   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

In [ ]:
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
modules = list(resnet.children())[:-1]   # drop fc
feature_extractor = nn.Sequential(*modules).to(device)
feature_extractor.eval()
for p in feature_extractor.parameters():
    p.requires_grad = False

def extract_embeddings(dataloader):
    embs, labs = [], []
    with torch.no_grad():
        for imgs, lbls in tqdm(dataloader, desc="Extracting embeddings"):
            imgs = imgs.to(device)
            feat = feature_extractor(imgs).view(imgs.size(0), -1)  # [B,512]
            embs.append(feat.cpu().numpy())
            labs.append(lbls.numpy())
    return np.vstack(embs), np.hstack(labs)

train_emb, train_lbl = extract_embeddings(train_loader_img)
val_emb, val_lbl     = extract_embeddings(val_loader_img)

print("Train emb shape:", train_emb.shape, " Val emb shape:", val_emb.shape)

In [ ]:
pca = PCA(n_components=pca_components)
train_emb_reduced = pca.fit_transform(train_emb)
val_emb_reduced   = pca.transform(val_emb)

def scale_to_angle(x):
    x_min, x_max = x.min(0), x.max(0)
    denom = (x_max - x_min)
    denom[denom == 0] = 1.0
    x_norm = (x - x_min) / denom
    return (x_norm - 0.5) * np.pi  # [-pi/2, pi/2]

train_inputs = scale_to_angle(train_emb_reduced).astype(np.float32)
val_inputs   = scale_to_angle(val_emb_reduced).astype(np.float32)

train_labels = train_lbl.astype(np.int64)
val_labels   = val_lbl.astype(np.int64)

train_tensor_ds = TensorDataset(torch.from_numpy(train_inputs), torch.from_numpy(train_labels))
val_tensor_ds   = TensorDataset(torch.from_numpy(val_inputs), torch.from_numpy(val_labels))

train_loader = DataLoader(train_tensor_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_tensor_ds, batch_size=batch_size, shuffle=False)


In [ ]:
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def vqc_circuit(inputs, weights):
    # inputs is a vector of length n_qubits (per sample)
    qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation='Y')
    qml.templates.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

# Define parameter shapes
weight_shapes = {
    "weights": qml.templates.StronglyEntanglingLayers.shape(
        n_layers=n_variational_layers, n_wires=n_qubits
    )
}

# TorchLayer (no batch_size in your version!)
qml_layer = TorchLayer(vqc_circuit, weight_shapes)


In [ ]:
class HybridVQCClassifier(nn.Module):
    def __init__(self, qlayer, n_qubits, n_classes=2):
        super().__init__()
        self.qlayer = qlayer
        self.fc = nn.Linear(n_qubits, n_classes)
    def forward(self, x):
        qout = self.qlayer(x)
        return self.fc(qout)

model_q = HybridVQCClassifier(qml_layer, n_qubits, num_classes).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_q.parameters(), lr=1e-3)

def evaluate(model, loader):
    model.eval()
    loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss += criterion(out, y).item() * x.size(0)
            preds = out.argmax(1)
            correct += (preds == y).sum().item()
            total += x.size(0)
    return loss/total, correct/total

for epoch in range(num_epochs):
    model_q.train()
    running_loss, correct, total = 0.0, 0, 0
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model_q(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += x.size(0)

    train_loss, train_acc = running_loss/total, correct/total
    val_loss, val_acc = evaluate(model_q, val_loader)
    print(f"Epoch {epoch+1}/{num_epochs} | Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")

torch.save(model_q.state_dict(), "hybrid_vqc_resnet18_cpu.pth")
print("Model saved: hybrid_vqc_resnet18_cpu.pth")

In [ ]:
# hybrid_end2end_vqc.py
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
import pennylane as qml
from pennylane.qnn import TorchLayer
from tqdm import tqdm

# ---------------------------
# Config
# ---------------------------
train_dir = "dataset/train"
val_dir   = "dataset/test"
batch_size = 16
epochs = 10
n_qubits = 12
n_layers = 3
num_classes = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------------------------
# Data transforms
# ---------------------------
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

train_ds = datasets.ImageFolder(train_dir, transform=transform)
val_ds   = datasets.ImageFolder(val_dir, transform=transform)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

# ---------------------------
# ResNet18 backbone (unfreeze last block)
# ---------------------------
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze everything first
for param in resnet.parameters():
    param.requires_grad = False

# Unfreeze last block (layer4) + fc (which we'll replace anyway)
for name, param in resnet.named_parameters():
    if "layer4" in name:
        param.requires_grad = True

# Remove the FC layer
modules = list(resnet.children())[:-1]
cnn = nn.Sequential(*modules)

# ---------------------------
# Quantum circuit (12 qubits)
# ---------------------------
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def vqc_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {
    "weights": qml.StronglyEntanglingLayers.shape(n_layers=n_layers, n_wires=n_qubits)
}

qml_layer = TorchLayer(vqc_circuit, weight_shapes)

# ---------------------------
# Hybrid Model
# ---------------------------
class HybridModel(nn.Module):
    def __init__(self, cnn, n_qubits, num_classes=2):
        super().__init__()
        self.cnn = cnn
        self.compress = nn.Linear(512, n_qubits)
        self.vqc = qml_layer
        self.fc = nn.Linear(n_qubits, num_classes)

    def forward(self, x):
        x = self.cnn(x)         # [B,512,1,1]
        x = x.view(x.size(0), -1)  # [B,512]
        x = self.compress(x)    # [B,n_qubits]
        q = self.vqc(x)         # [B,n_qubits]
        out = self.fc(q)        # [B,num_classes]
        return out

model = HybridModel(cnn, n_qubits, num_classes).to(device)

# ---------------------------
# Loss & Optimizer
# ---------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

# ---------------------------
# Training & Validation
# ---------------------------
def evaluate(model, loader):
    model.eval()
    loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            loss += criterion(out, labels).item() * imgs.size(0)
            preds = out.argmax(1)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)
    return loss/total, correct/total

for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0, 0, 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total += imgs.size(0)

    train_loss, train_acc = running_loss/total, correct/total
    val_loss, val_acc = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}/{epochs} | Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")

torch.save(model.state_dict(), "hybrid_end2end_vqc.pth")
print("✅ Model saved: hybrid_end2end_vqc.pth")
